In [1]:
import ROOT
import numpy as np
import matplotlib.pyplot as plt


/home/leoperes/.local/lib/python3.12/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


In [2]:

f_list = ["../np02vd_raw_run039510_0000_df-s04-d0_dw_0_20250919T123428_0kV_gallery.root",
"../np02vd_raw_run039510_0001_df-s04-d0_dw_0_20250919T123526_0kV_gallery.root",
"../np02vd_raw_run039510_0002_df-s04-d0_dw_0_20250919T123624_0kV_gallery.root",
"../np02vd_raw_run039510_0003_df-s04-d0_dw_0_20250919T123722_0kV_gallery.root",
"../np02vd_raw_run039510_0004_df-s04-d0_dw_0_20250919T123821_0kV_gallery.root",
"../np02vd_raw_run039510_0005_df-s04-d0_dw_0_20250919T123919_0kV_gallery.root"]


chain = ROOT.TChain("WaveformTree")


for f_in in f_list:
    chain.Add(f_in)

n_entries = chain.GetEntries()
print(n_entries)


8292065


In [3]:
# -------------------------------------------------------
# Configuration
# -------------------------------------------------------
channels_dict_membrana = {
    "CH_M1_1": 2010, 
    "CH_M1_2": 2011, 
    "CH_M2_1": 2020, 
    "CH_M2_2": 2021, 
    "CH_M3_1": 2030, 
    "CH_M3_2": 2031, 
    "CH_M4_1": 2040, 
    "CH_M4_2": 2041, 
    "CH_M5_1": 2050, 
    "CH_M5_2": 2051, 
    "CH_M6_1": 2060, 
    "CH_M6_2": 2061,
    "CH_M7_1": 2070,
    "CH_M7_2": 2071,
    "CH_M8_1": 2080,
    "CH_M8_2": 2081
}

channels_dict_cathode = {
    "CH_C1_1": 1010, 
    "CH_C1_2": 1011, 
    "CH_C2_1": 1020, 
    "CH_C2_2": 1021, 
    "CH_C3_1": 1030, 
    "CH_C3_2": 1031, 
    "CH_C4_1": 1040, 
    "CH_C4_2": 1041,
    "CH_C5_1": 1050,
    "CH_C5_2": 1051,
    "CH_C6_1": 1060,
    "CH_C6_2": 1061,
    "CH_C7_1": 1070,
    "CH_C7_2": 1071,
    "CH_C8_1": 1080,
    "CH_C8_2": 1081
}

# Use 1 if one ADC sample corresponds to one timestamp tick.
# Keep the coincidence logic in timestamp units unless you have
# confirmed the timestamp conversion in the decoder source.
SAMPLE_TICKS = 1

# Initial coincidence window, in timestamp ticks.
# If 1 tick = 16 ns, then 125 ticks = 2 us.
COINCIDENCE_WINDOW_TICKS = 20

def baseline(waveform):
    """Calculate the baseline of a waveform."""
    return np.mean(waveform[0:300])

def signal_amplitude(waveform):
    waveform = np.asarray(waveform)
    signal_region = waveform[30:130]
    return np.max(signal_region) - np.min(signal_region)

def signal_peak_tick(waveform):
    waveform = np.asarray(waveform)
    signal_region = waveform[30:130]
    return 30 + int(np.argmax(signal_region))

def noise_rms(waveform):
    """Calculate the RMS of the noise in a waveform."""
    return np.std(waveform[0:300])

def preSignalAmplitude(waveform):
    """Calculate the pre-signal amplitude of a waveform."""
    return np.max(waveform[0:300]) - np.min(waveform[0:300])

def postSignalAmplitude(waveform):
    """Calculate the post-signal amplitude of a waveform."""
    return np.max(waveform[350:1023]) - np.min(waveform[350:1023])

def Waveforms_Coincidence_byAmplitude(waveform_1, waveform_2, timestamp_1, timestamp_2, threshold):
    """Check if two waveforms are in coincidence based on amplitude threshold."""
    amp1 = signal_amplitude(waveform_1)
    amp2 = signal_amplitude(waveform_2)
    tick_amp1 = signal_peak_tick(waveform_1)
    tick_amp2 = signal_peak_tick(waveform_2)


    if amp1 > threshold and amp2 > threshold and abs(tick_amp1 - tick_amp2) <= COINCIDENCE_WINDOW_TICKS and timestamp_1 == timestamp_2:
        return True
    else:
        return False

def Waveforms_Coincidence_byTimestamp(waveform_1, waveform_2, timestamp_1, timestamp_2):
    """Check if two waveforms are in coincidence based on timestamp."""
    tick_amp1 = signal_peak_tick(waveform_1)
    tick_amp2 = signal_peak_tick(waveform_2)

    if abs(tick_amp1 - tick_amp2) <= COINCIDENCE_WINDOW_TICKS and timestamp_1 == timestamp_2:
        return True
    else:
        return False

In [4]:
# Read only needed branches
chain.SetBranchStatus("*", 0)
chain.SetBranchStatus("channel", 1)
chain.SetBranchStatus("adc", 1)
chain.SetBranchStatus("timestamp", 1)
chain.SetBranchStatus("event", 1)
chain.SetBranchStatus("waveform_index", 1)

df = ROOT.RDataFrame(chain)

In [5]:


valid_channels = np.array(sorted(channels_dict_membrana.values()), dtype=np.int64)
channel_evt_filter = (
    "(" + " || ".join(f"channel == {int(ch)}" for ch in valid_channels) + ")"
    " && event < 1000"
)

print(channel_evt_filter)
arr = (
    df.Filter(channel_evt_filter)
      .AsNumpy(["event", "channel", "timestamp", "waveform_index","adc"])
)




events = arr["event"].astype(np.int64, copy=False)
wf = arr["adc"]
wf_index = arr["waveform_index"].astype(np.int64, copy=False)
channel_arr = arr["channel"].astype(np.int64, copy=False)
timestamp_arr = arr["timestamp"].astype(np.int64, copy=False)

print(f"Total waveforms collected: {len(events)}")


(channel == 2010 || channel == 2011 || channel == 2020 || channel == 2021 || channel == 2030 || channel == 2031 || channel == 2040 || channel == 2041 || channel == 2050 || channel == 2051 || channel == 2060 || channel == 2061 || channel == 2070 || channel == 2071 || channel == 2080 || channel == 2081) && event < 1000
Total waveforms collected: 486371


In [9]:
mask = (channel_arr == 2030) & (events == 90)
print(timestamp_arr[mask])

[109892829527164944 109892829527189648 109892829527202320
 109892829527208064 109892829527209600 109892829527254768
 109892829527270688 109892829527291248 109892829527295904
 109892829527336608 109892829527352528 109892829527357792
 109892829527385088 109892829527392288 109892829527393696
 109892829527417280 109892829527428160 109892829527450528
 109892829527477232 109892829527503888 109892829527508640
 109892829527517520 109892829527530864 109892829527538512
 109892829527552240 109892829527571936 109892829527613424
 109892829527619392 109892829527664496 109892829527696432
 109892829527725536]


In [22]:
timestamps_2030 = timestamp_arr[
    (channel_arr == 2030) & (events == 100)
].astype(np.int64)

timestamps_2060 = timestamp_arr[
    (channel_arr == 2060) & (events == 100)
].astype(np.int64)

In [11]:
import numpy as np


def match_waveform_starts(
    times_a,
    times_b,
    window_ticks=1,
    tick_ns=16,
):
    """
    One-to-one matching of waveform start timestamps.

    Returns tuples:
        (index_a, index_b, time_a, time_b, delta_ns)
    """

    times_a = np.asarray(times_a, dtype=np.int64)
    times_b = np.asarray(times_b, dtype=np.int64)

    order_a = np.argsort(times_a)
    order_b = np.argsort(times_b)

    sorted_a = times_a[order_a]
    sorted_b = times_b[order_b]

    window_ns = window_ticks * tick_ns
    candidates = []

    for ia, time_a in enumerate(sorted_a):
        left = np.searchsorted(
            sorted_b,
            time_a - window_ns,
            side="left",
        )
        right = np.searchsorted(
            sorted_b,
            time_a + window_ns,
            side="right",
        )

        for ib in range(left, right):
            delta_ns = int(sorted_b[ib] - time_a)

            candidates.append(
                (
                    abs(delta_ns),
                    ia,
                    ib,
                    delta_ns,
                )
            )

    # Match the closest pairs first.
    candidates.sort(key=lambda item: item[0])

    used_a = set()
    used_b = set()
    matches = []

    for _, ia, ib, delta_ns in candidates:
        if ia in used_a or ib in used_b:
            continue

        used_a.add(ia)
        used_b.add(ib)

        original_a = int(order_a[ia])
        original_b = int(order_b[ib])

        matches.append(
            (
                original_a,
                original_b,
                int(sorted_a[ia]),
                int(sorted_b[ib]),
                delta_ns,
            )
        )

    return matches

In [26]:
matches = match_waveform_starts(
    timestamps_2030,
    timestamps_2060,
    window_ticks=10,
)

print(f"Channel 2030 waveforms: {len(timestamps_2030)}")
print(f"Channel 2060 waveforms: {len(timestamps_2060)}")
print(f"Matched waveform starts: {len(matches)}")

for original_2030, original_2060, t2030, t2060, dt_ns in matches:
    print(
        f"original_2030: {original_2030}, original_2060: {original_2060}, "
        f"2030: {t2030}  "
        f"2060: {t2060}  "
        f"dt: {dt_ns:+d} ns"
    )

Channel 2030 waveforms: 33
Channel 2060 waveforms: 33
Matched waveform starts: 4
original_2030: 1, original_2060: 2, 2030: 109892829558422032  2060: 109892829558422032  dt: +0 ns
original_2030: 3, original_2060: 4, 2030: 109892829558458816  2060: 109892829558458816  dt: +0 ns
original_2030: 27, original_2060: 25, 2030: 109892829558936288  2060: 109892829558936288  dt: +0 ns
original_2030: 14, original_2060: 16, 2030: 109892829558748432  2060: 109892829558748448  dt: +16 ns


In [24]:
for window_ticks in [0, 1, 2, 5,10]:
    matches = match_waveform_starts(
        timestamps_2030,
        timestamps_2060,
        window_ticks=window_ticks,
    )

    print(
        f"Window ±{window_ticks} ticks "
        f"(±{window_ticks * 16} ns): "
        f"{len(matches)} matches"
    )

Window ±0 ticks (±0 ns): 3 matches
Window ±1 ticks (±16 ns): 4 matches
Window ±2 ticks (±32 ns): 4 matches
Window ±5 ticks (±80 ns): 4 matches
Window ±10 ticks (±160 ns): 4 matches


In [25]:
matches = match_waveform_starts(
    timestamps_2030,
    timestamps_2060,
    window_ticks=1,
)

fraction_2030 = len(matches) / len(timestamps_2030)
fraction_2060 = len(matches) / len(timestamps_2060)

symmetric_fraction = (
    2 * len(matches)
    / (len(timestamps_2030) + len(timestamps_2060))
)

print("Fraction of 2030 matched:", fraction_2030)
print("Fraction of 2060 matched:", fraction_2060)
print("Symmetric matched fraction:", symmetric_fraction)

Fraction of 2030 matched: 0.12121212121212122
Fraction of 2060 matched: 0.12121212121212122
Symmetric matched fraction: 0.12121212121212122


In [15]:
waveform_coincidence = (
    fraction_2030 >= 0.90
    and fraction_2060 >= 0.90
)